In [54]:
import os
import pandas as pd
import statsmodels.api as sm
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])

# __I. Estimation__

## A. Mean Forecasts

In [55]:
a = '2022-12-31'
mean_spf_trim = mean_spf_trim.loc[(mean_spf_trim['DATE'] >= '1981-12-31') & (mean_spf_trim['DATE'] <= a)]  # Filter data

def est_table(df):
    est_table = pd.DataFrame(index=['const', 'beta_1'])
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        reg = initial.get_robustcov_results(cov_type='HAC', maxlags=1, hasconst=True)
        est_table[f'coef_t{j}'] = reg.params
        est_table[f'std_err{j}'] = reg.bse
        est_table[f'tval{j}'] = reg.tvalues
        est_table[f'pval{j}'] = reg.pvalues
        est_table[f'nobs{j}'] = initial.nobs
    return est_table

mean_estimations = est_table(mean_spf_trim)

print(mean_estimations)

         coef_t0  std_err0     tval0     pval0  nobs0   coef_t1  std_err1  \
const   0.000121  0.000202  0.596384  0.551746  165.0  0.000227  0.000626   
beta_1  0.641148  0.129187  4.962962  0.000002  165.0  0.612336  0.209225   

           tval1     pval1  nobs1   coef_t2  std_err2     tval2     pval2  \
const   0.362555  0.717407  165.0  0.000149  0.001089  0.137187  0.891052   
beta_1  2.926693  0.003915  165.0  0.605728  0.279131  2.170052  0.031451   

        nobs2   coef_t3  std_err3     tval3     pval3  nobs3  
const   165.0 -0.000042  0.001494 -0.028219  0.977522  165.0  
beta_1  165.0  0.686962  0.312256  2.199996  0.029214  165.0  


## B. Individual Forecasts

In [56]:
### Pooled OLS ###
ind_spf_trim = ind_spf_trim.loc[(ind_spf_trim['DATE'] >= '1981-12-31') & (ind_spf_trim['DATE'] <= a)]  # Filter data

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')
ind_est_pld = est_table(ind_spf_trim)

print(ind_est_pld)

         coef_t0  std_err0     tval0     pval0   nobs0   coef_t1  std_err1  \
const   0.000137  0.000082  1.672707  0.094457  4354.0  0.000166  0.000178   
beta_1  0.054849  0.033152  1.654493  0.098099  4354.0 -0.008185  0.047573   

           tval1     pval1   nobs1   coef_t2  std_err2     tval2     pval2  \
const   0.931115  0.351845  4354.0  0.000067  0.000259  0.257706  0.796646   
beta_1 -0.172045  0.863410  4354.0 -0.053317  0.050845 -1.048621  0.294411   

         nobs2   coef_t3  std_err3     tval3     pval3   nobs3  
const   4354.0 -0.000145  0.000332 -0.436232  0.662690  4354.0  
beta_1  4354.0 -0.035625  0.054221 -0.657036  0.511193  4354.0  


In [57]:
### ID Fixed Effects ###
def est_table_fe(df):
    est_table = pd.DataFrame(index=['const', 'beta_1'])
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        reg = initial.get_robustcov_results(cov_type='HAC', maxlags=1, hasconst=True)
        est_table[f'coef_t{j}'] = reg.params[:2]
        est_table[f'std_err{j}'] = reg.bse[:2]
        est_table[f'tval{j}'] = reg.tvalues[:2]
        est_table[f'pval{j}'] = reg.pvalues[:2]
        est_table[f'nobs{j}'] = initial.nobs
    return est_table

ind_est_fe = est_table_fe(ind_spf_trim)

print(ind_est_fe)

         coef_t0  std_err0    tval0     pval0   nobs0   coef_t1  std_err1  \
const   0.001168  0.003664  0.31872  0.749955  4354.0 -0.005281  0.006986   
beta_1  0.043438  0.033326  1.30340  0.192511  4354.0 -0.047637  0.046668   

           tval1     pval1   nobs1   coef_t2  std_err2     tval2     pval2  \
const  -0.755902  0.449751  4354.0 -0.021585  0.006331 -3.409474  0.000657   
beta_1 -1.020763  0.307426  4354.0 -0.118037  0.047868 -2.465895  0.013707   

         nobs2   coef_t3  std_err3      tval3          pval3   nobs3  
const   4354.0 -0.030471  0.000984 -30.970226  1.281018e-189  4354.0  
beta_1  4354.0 -0.125053  0.048484  -2.579271   9.935065e-03  4354.0  


In [58]:
### Two-way Fixed Effects ###
def est_table_fe2(df):
    est_table = pd.DataFrame(index=['const', 'beta_1'])
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float), pd.get_dummies(ind_spf_trim['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        reg = initial.get_robustcov_results(cov_type='HAC', maxlags=1, hasconst=True)
        est_table[f'coef_t{j}'] = reg.params[:2]
        est_table[f'std_err{j}'] = reg.bse[:2]
        est_table[f'tval{j}'] = reg.tvalues[:2]
        est_table[f'pval{j}'] = reg.pvalues[:2]
        est_table[f'nobs{j}'] = initial.nobs
    return est_table

ind_est_fe2 = est_table_fe2(ind_spf_trim)

print(ind_est_fe2)

         coef_t0  std_err0      tval0          pval0   nobs0   coef_t1  \
const  -0.007012  0.001181  -5.936386   3.162436e-09  4354.0 -0.020997   
beta_1 -0.621496  0.019961 -31.135931  1.135539e-190  4354.0 -0.576168   

        std_err1      tval1          pval1   nobs1   coef_t2  std_err2  \
const   0.001548 -13.560981   5.480156e-41  4354.0 -0.028276  0.002344   
beta_1  0.022936 -25.120843  2.175368e-129  4354.0 -0.530349  0.026061   

            tval2         pval2   nobs2   coef_t3  std_err3      tval3  \
const  -12.062041  6.248185e-33  4354.0 -0.030081  0.003800  -7.915410   
beta_1 -20.350101  1.160659e-87  4354.0 -0.490592  0.028189 -17.403918   

               pval3   nobs3  
const   3.169952e-15  4354.0  
beta_1  1.928975e-65  4354.0  


## C. AR(1) Estimations

In [59]:
from statsmodels.tsa.ar_model import AutoReg
import matplotlib.pyplot as plt

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

vintage_trim = vintage_trim.loc[(vintage_trim['DATE'] >= '1965-06-30') & (vintage_trim['DATE'] <= a)]  # Filter data
vintage_trim = vintage_trim.set_index('DATE')

def ar_j(df):
    ar_table = pd.DataFrame()
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    ar_table.loc['const', 'coef'] = reg.params[0]
    ar_table.loc['phi', 'coef'] = reg.params[1]
    ar_table.loc['rho', 'coef'] = reg.roots
    return ar_table

ar_table = ar_j(vintage_trim)

print(ar_table)

           coef
const  0.001628
phi    0.960717
rho    1.040889


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency Q-DEC will be used.
  self._init_dates(dates, freq)


# __II. Model Parameters__

In [60]:
def params(dfm, dfp, dff, dff2, ar):
    params = pd.DataFrame(index=['ols', 'pld', 'fe', 'fe2'])
    params.loc['ols', 'lambda'] = dfm.loc['beta_1', 'coef_t3'] / (1 + dfm.loc['beta_1', 'coef_t3'])
    params.loc['ols', 'G'] = 1 / (1 + dfm.loc['beta_1', 'coef_t3'])
    pldc = dfp.iloc[1]['coef_t3']
    fec = dff.iloc[1]['coef_t3']
    fe2c = dff2.iloc[1]['coef_t3']
    ar1 = ar.loc['const', 'coef']
    params.loc['pld', 'Theta'] = (-((2 * pldc) + 1) + np.sqrt(((2 * pldc) + 1)**2 - 4 * (pldc + (pldc * ar1**2) + 1) * pldc)) / (2 * (pldc + (pldc * ar1**2) + 1))
    params.loc['fe', 'Theta'] = (-((2 * fec) + 1) + np.sqrt(((2 * fec) + 1)**2 - 4 * (fec + (fec * ar1**2) + 1) * fec)) / (2 * (fec + (fec * ar1**2) + 1))
    params.loc['fe2', 'Theta'] = (-((2 * fe2c) + 1) + np.sqrt(((2 * fe2c) + 1)**2 - 4 * (fe2c + (fe2c * ar1**2) + 1) * fe2c)) / (2 * (fe2c + (fe2c * ar1**2) + 1))
    return params

parameters = params(mean_estimations, ind_est_pld, ind_est_fe, ind_est_fe2, ar_table)

print(parameters)

       lambda         G     Theta
ols  0.407218  0.592782       NaN
pld       NaN       NaN  0.036941
fe        NaN       NaN  0.142926
fe2       NaN       NaN  0.963064


# __III. CSV Export (Optional)__

In [61]:
mean_estimations.to_csv('output/mean_estimations.csv', sep=',', index=True)
ind_est_pld.to_csv('output/ind_est_pld.csv', sep=',', index=True)
ind_est_fe.to_csv('output/ind_est_fe.csv', sep=',', index=True)
ind_est_fe2.to_csv('output/ind_est_fe2.csv', sep=',', index=True)
ar_table.to_csv('output/ar_estimations.csv', sep=',', index=True)
parameters.to_csv('output/parameters.csv', sep=',', index=True)